# Phase_1

# 1. Data Collection:

In [ ]:
import kagglehub

path = kagglehub.dataset_download("pariza/bbc-news-summary")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd

def read_file(file_path):
    encodings = ['utf-8', 'latin-1', 'cp1252']

    for enc in encodings:
        try:
            with open(file_path, 'r', encoding=enc) as f:
                return f.read()
        except:
            continue
    return ""

all_files = []
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".txt"):
            all_files.append(os.path.join(root, file))

print("Total Documents in Dataset:", len(all_files))


In [ ]:
MAX_DOCS = 350
data = []
count = 0

for root, dirs, files in os.walk(path):
    category = os.path.basename(root)
    for file in files:
        if file.endswith(".txt") and count < MAX_DOCS:
            file_path = os.path.join(root, file)
            text = read_file(file_path)

            if text.strip():
                data.append({
                    "document ID": file,
                    "category": category,
                    "content": text.strip()
                })
                count += 1

df_corpus = pd.DataFrame(data)
df_corpus["length"] = df_corpus["content"].apply(lambda x: len(x.split()))
df_corpus["processed_content"] = df_corpus["content"].apply(lambda x: preprocess(x))
df_corpus.to_csv("bbc.csv", index=False, encoding='utf-8-sig')
print(f"Total documents processed: {len(df_corpus)}")
print("CSV file created successfully")
df_corpus.head()

In [ ]:
df = df_corpus.copy()
df["length"] = df["content"].apply(lambda x: len(x.split()))
df.to_csv("bbc.csv", index=False, encoding='utf-8-sig')
print("CSV file successfully ")

# 2. Preprocessing

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords', quiet=True)
STOP_WORDS = set(stopwords.words('english'))
STEMMER = PorterStemmer()
CLEAN_REGEX = re.compile(r'[^a-z\s]+')

In [ ]:
def preprocess(text, remove_stopwords=True, stemming=True):
    text = text.lower().strip()
    text = CLEAN_REGEX.sub(' ', text)
    tokens = text.split()
    if remove_stopwords:
        tokens = [w for w in tokens if w not in STOP_WORDS and len(w) > 2]
    if stemming:
        tokens = [STEMMER.stem(w) for w in tokens]
    return tokens

In [ ]:
for i in range(5):
    print(f"\n Document {i+1}:")
    original_text = df_corpus['content'].iloc[i]
    print("Original:", original_text[:200])
    processed = preprocess(original_text)
    print("Processed:", processed[:20])

# 3. Indexing

In [ ]:
import pandas as pd

df = pd.read_csv("bbc.csv")
df["tokens"] = df["content"].apply(preprocess)
print("Data Indexing")
df.head()

In [ ]:
from collections import defaultdict

inverted_index = defaultdict(dict)

for _, row in df.iterrows():
    doc_id = row["document ID"]
    tokens = row["tokens"]

    for word in tokens:
        if doc_id not in inverted_index[word]:
            inverted_index[word][doc_id] = 0

        inverted_index[word][doc_id] += 1

print("Inverted Index Successfully")

In [ ]:
for term in inverted_index:
    inverted_index[term] = dict(sorted(inverted_index[term].items()))

In [ ]:
for i, (term, postings) in enumerate(inverted_index.items()):
    print(f"\n Term: {term}")
    print("Postings:", postings)

    if i == 5:
        break